<a href="https://colab.research.google.com/github/nemitha21/S-P500_Stock_Analysis_CWK1/blob/main/S%26P500_Stock_Analysis_CWK1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import io, json, os, platform
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import requests
import yfinance as yf

# Folders (commit these to your public GitHub repo)
os.makedirs("data/raw", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

In [ ]:
START = "2021-01-01"
END = "2025-01-01"          # yfinance treats END as EXCLUSIVE, as the brief requires
BENCHMARK = "^GSPC"
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

In [ ]:
RETRIEVAL_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

# Wikipedia can reject the default pandas user agent, so fetch the page first
html = requests.get(WIKI_URL, headers={"User-Agent": "Mozilla/5.0 (coursework; student)"}, timeout=30).text
tables = pd.read_html(io.StringIO(html))
constituents = tables[0].copy()          # first table = current constituents

constituents = constituents.rename(columns={
    "Symbol": "wiki_symbol",
    "Security": "company",
    "GICS Sector": "sector",
    "GICS Sub-Industry": "sub_industry",
    "Date added": "date_added",
})
# Yahoo uses "-" where Wikipedia uses "." (e.g. BRK.B -> BRK-B)
constituents["ticker"] = constituents["wiki_symbol"].str.replace(".", "-", regex=False)
constituents["retrieved_utc"] = RETRIEVAL_UTC

print("Retrieved:", RETRIEVAL_UTC)
print("Rows:", len(constituents), "| unique tickers:", constituents["ticker"].nunique())
print(constituents["sector"].value_counts())

constituents.to_csv("data/raw/sp500_constituents_wikipedia.csv", index=False)

Retrieved: 2026-11-18 15:22:48 UTC
Rows: 503 | unique tickers: 503
sector
Industrials               83
Information Technology    76
Financials                76
Health Care               60
Consumer Discretionary    47
Consumer Staples          33
Utilities                 31
Real Estate               30
Materials                 24
Communication Services    22
Energy                    21
Name: count, dtype: int64


In [ ]:
tickers = sorted(constituents["ticker"].unique().tolist())

# auto_adjust=False keeps BOTH Close and Adj Close (needed: adjusted for returns, Close for Monetary)
# actions=True adds Dividends and Stock Splits columns (needed for the corporate action checks in 1.2)
def download(tkrs):
    return yf.download(
        tkrs, start=START, end=END, interval="1d",
        auto_adjust=False, actions=True,
        group_by="column", threads=True, progress=False,
    )

raw = download(tickers + [BENCHMARK])
print(raw.shape)
print(raw.columns.get_level_values(0).unique().tolist())

ERROR:yfinance:
5 Failed downloads:
ERROR:yfinance:['SNDK', 'VYLR', 'HONA', 'FDXF', 'Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


(1005, 4022)
['Adj Close', 'Close', 'Dividends', 'High', 'Low', 'Open', 'Stock Splits', 'Volume']


In [ ]:
fields = ["Open", "High", "Low", "Close", "Adj Close", "Volume", "Dividends", "Stock Splits"]

def failed_list(df, tkrs):
    ok = df["Adj Close"].notna().any()
    return [t for t in tkrs if (t not in ok.index) or (not ok[t])]

failed = failed_list(raw, tickers + [BENCHMARK])
print("Failed first pass:", failed)

retry_log = []
for t in failed:
    try:
        d = download([t])
        if "Adj Close" in d.columns.get_level_values(0) and d["Adj Close"].notna().any().any():
            for f in fields:
                if f in d.columns.get_level_values(0):
                    raw[(f, t)] = d[f][t] if isinstance(d[f], pd.DataFrame) else d[f]
            retry_log.append((t, "recovered"))
        else:
            retry_log.append((t, "no data"))
    except Exception as e:
        retry_log.append((t, f"error: {e}"))

raw = raw.sort_index(axis=1)
still_failed = failed_list(raw, tickers + [BENCHMARK])
print("Retry log:", retry_log)
print("Still failed after retry:", still_failed)


Failed first pass: ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['FDXF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['HONA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['SNDK']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
ERROR:yfinance:
1 Failed download:
ERRO

Retry log: [('FDXF', 'no data'), ('HONA', 'no data'), ('Q', 'no data'), ('SNDK', 'no data'), ('VYLR', 'no data')]
Still failed after retry: ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']


In [ ]:
for f in fields:
    if f in raw.columns.get_level_values(0):
        raw[f].to_csv(f"data/raw/{f.replace(' ', '_').lower()}.csv")

# Benchmark on its own, for clarity
bench = pd.DataFrame({f: raw[f][BENCHMARK] for f in ["Close", "Adj Close", "Volume"]})
bench.to_csv("data/raw/gspc_benchmark.csv")

# Download log (feeds your inclusion/exclusion log in 1.2)
download_log = pd.DataFrame({
    "ticker": tickers + [BENCHMARK],
    "downloaded": [t not in still_failed for t in tickers + [BENCHMARK]],
})
download_log.to_csv("outputs/download_log.csv", index=False)

In [ ]:
meta = {
    "wikipedia_url": WIKI_URL,
    "retrieval_utc": RETRIEVAL_UTC,
    "data_source": "yfinance (Yahoo Finance), auto_adjust=False, actions=True",
    "start_inclusive": START,
    "end_exclusive": END,
    "benchmark": BENCHMARK,
    "n_tickers_requested": len(tickers),
    "n_tickers_failed": len(still_failed),
    "failed_tickers": still_failed,
    "first_date_in_data": str(raw.index.min().date()),
    "last_date_in_data": str(raw.index.max().date()),
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "yfinance": yf.__version__,
}
with open("data/raw/acquisition_metadata.json", "w") as fh:
    json.dump(meta, fh, indent=2)
print(json.dumps(meta, indent=2))

{
  "wikipedia_url": "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
  "retrieval_utc": "2026-11-18 15:22:48 UTC",
  "data_source": "yfinance (Yahoo Finance), auto_adjust=False, actions=True",
  "start_inclusive": "2021-01-01",
  "end_exclusive": "2025-01-01",
  "benchmark": "^GSPC",
  "n_tickers_requested": 503,
  "n_tickers_failed": 5,
  "failed_tickers": [
    "FDXF",
    "HONA",
    "Q",
    "SNDK",
    "VYLR"
  ],
  "first_date_in_data": "2021-01-04",
  "last_date_in_data": "2024-12-31",
  "python": "3.13.18",
  "pandas": "2.2.3",
  "numpy": "2.4.6",
  "yfinance": "0.2.66"
}


In [ ]:
print("Date range:", raw.index.min().date(), "to", raw.index.max().date())
print("Trading days:", len(raw))
print("Benchmark rows with Adj Close:", raw["Adj Close"][BENCHMARK].notna().sum())
print("Tickers with data:", raw["Adj Close"].notna().any().sum() - 1, "(excluding benchmark)")

Date range: 2021-01-04 to 2024-12-31
Trading days: 1005
Benchmark rows with Adj Close: 1005
Tickers with data: 498 (excluding benchmark)


In [9]:
#1.2 Validation
import json, os
import numpy as np
import pandas as pd

os.makedirs("data/clean", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

BENCH = "^GSPC"

def load(name):
    return pd.read_csv(f"data/raw/{name}.csv", index_col=0, parse_dates=True)

adj, close, vol = load("adj_close"), load("close"), load("volume")
div, spl = load("dividends"), load("stock_splits")
cons = pd.read_csv("data/raw/sp500_constituents_wikipedia.csv")
meta = json.load(open("data/raw/acquisition_metadata.json"))

In [10]:
failed = meta["failed_tickers"]
print(cons.loc[cons.ticker.isin(failed), ["ticker", "company", "sector", "date_added"]])

    ticker              company                  sector  date_added
192   FDXF        FedEx Freight             Industrials  2026-06-01
233   HONA  Honeywell Aerospace             Industrials  2026-06-29
379      Q    Qnity Electronics  Information Technology  2025-11-03
402   SNDK              Sandisk  Information Technology  2025-11-28
478   VYLR                Vylor        Consumer Staples  2026-10-01


In [11]:
print("Duplicate dates:", int(adj.index.duplicated().sum()))
print("Index sorted:", adj.index.is_monotonic_increasing)
print("All price/volume tables share the same dates:",
      all(adj.index.equals(x.index) for x in (close, vol, div, spl)))

ref_dates = adj.index[adj[BENCH].notna()]          # benchmark trading calendar
N_REF, REF_FIRST, REF_LAST = len(ref_dates), ref_dates.min(), ref_dates.max()
print("Benchmark trading days:", N_REF, "|", REF_FIRST.date(), "to", REF_LAST.date())

Duplicate dates: 0
Index sorted: True
All price/volume tables share the same dates: True
Benchmark trading days: 1005 | 2021-01-04 to 2024-12-31


In [12]:
SPLIT_JUMP = 0.40      # |daily Close return| above this with no recorded split -> review flag
RATIO_TOL = 1e-3       # Adj Close / Close ratio change with no dividend -> review flag

tickers = [t for t in adj.columns if t != BENCH]
rows = []
for t in tickers + [BENCH]:
    a, c, v = adj[t], close[t], vol[t]
    valid = a.dropna()
    r = {"ticker": t, "n_valid_adj": len(valid)}
    if len(valid) == 0:
        rows.append(r); continue

    first, last = valid.index.min(), valid.index.max()
    win = slice(first, last)
    r.update({
        "first_date": first, "last_date": last,
        "n_missing_vs_calendar": N_REF - len(valid),
        # missing values INSIDE the share's own history (gaps), across all three fields
        "internal_missing": int(pd.concat([a.loc[win], c.loc[win], v.loc[win]], axis=1).isna().any(axis=1).sum()),
        "nonpositive_price": int((a <= 0).sum() + (c <= 0).sum()),
        "negative_volume": int((v < 0).sum()),
        "zero_volume_days": int((v == 0).sum()),
        "n_splits": int((spl[t].fillna(0) > 0).sum()),
        "n_dividends": int((div[t].fillna(0) > 0).sum()),
    })

    # --- corporate action checks ---
    ret = c.pct_change(fill_method=None)
    r["unexplained_big_jumps"] = int(((ret.abs() > SPLIT_JUMP) & (spl[t].fillna(0) == 0)).sum())

    ratio = a / c
    rchg = ratio.pct_change(fill_method=None).abs()
    r["unexplained_adj_ratio_changes"] = int(((rchg > RATIO_TOL) & (div[t].fillna(0) == 0) & (spl[t].fillna(0) == 0)).sum())

    # traded value (Close x Volume) before vs after each split: ratio near 1 = price and volume on a compatible basis
    tv = c * v
    sp_dates = spl.index[spl[t].fillna(0) > 0]
    tv_ratios = []
    for d in sp_dates:
        i = tv.index.get_loc(d)
        before, after = tv.iloc[max(i - 5, 0):i].mean(), tv.iloc[i:i + 5].mean()
        tv_ratios.append(round(after / before, 2) if before and before > 0 else np.nan)
    r["split_dates"] = ";".join(str(d.date()) for d in sp_dates)
    r["split_traded_value_ratio"] = ";".join(map(str, tv_ratios))
    rows.append(r)

summary = pd.DataFrame(rows).set_index("ticker")
summary.to_csv("outputs/validation_summary.csv")
print("Benchmark row:\n", summary.loc[BENCH].to_string())

Benchmark row:
 n_valid_adj                                     1005
first_date                       2021-01-04 00:00:00
last_date                        2024-12-31 00:00:00
n_missing_vs_calendar                            0.0
internal_missing                                 0.0
nonpositive_price                                0.0
negative_volume                                  0.0
zero_volume_days                                 1.0
n_splits                                         0.0
n_dividends                                      0.0
unexplained_big_jumps                            0.0
unexplained_adj_ratio_changes                    0.0
split_dates                                         
split_traded_value_ratio                            


In [13]:
s = summary.drop(index=BENCH)
cov = s["n_valid_adj"].fillna(0) / N_REF
print("Shares with 100% coverage     :", int((cov == 1).sum()))
print("Shares with >=99% coverage    :", int((cov >= 0.99).sum()))
print("Shares with >=95% coverage    :", int((cov >= 0.95).sum()))
print("Shares with >=90% coverage    :", int((cov >= 0.90).sum()))
print("\nMissing-day distribution (among shares with data):")
print(s["n_missing_vs_calendar"].describe())
print("\nLate listings (first date after benchmark start):")
late = s[s["first_date"] > REF_FIRST]
print(late[["first_date", "n_valid_adj"]].sort_values("first_date").to_string())


Shares with 100% coverage     : 488
Shares with >=99% coverage    : 488
Shares with >=95% coverage    : 489
Shares with >=90% coverage    : 491

Missing-day distribution (among shares with data):
count    498.000000
mean       8.405622
std       74.461529
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max      812.000000
Name: n_missing_vs_calendar, dtype: float64

Late listings (first date after benchmark start):
       first_date  n_valid_adj
ticker                        
EXE    2021-02-10          979
COIN   2021-04-14          936
APP    2021-04-15          935
HOOD   2021-07-29          862
CEG    2022-01-19          742
GEHC   2022-12-15          513
VLTO   2023-10-04          313
RDDT   2024-03-21          197
SOLV   2024-03-26          194
GEV    2024-03-27          193


In [14]:
def reasons(r):
    out = []
    if r["n_valid_adj"] == 0 or pd.isna(r.get("first_date")):
        return ["no_price_data"]
    if r["first_date"] > REF_FIRST or r["last_date"] < REF_LAST:
        out.append("unequal_history")
    if r["internal_missing"] > 0:
        out.append("missing_values")
    if r["nonpositive_price"] > 0:
        out.append("nonpositive_price")
    if r["negative_volume"] > 0:
        out.append("negative_volume")
    if r["zero_volume_days"] > 0:
        out.append("zero_volume_days")
    return out

s = s.copy()
s["exclusion_reasons"] = s.apply(reasons, axis=1)
s["included"] = s["exclusion_reasons"].apply(len) == 0

excl = s.loc[~s["included"], ["exclusion_reasons"]].copy()
excl["exclusion_reasons"] = excl["exclusion_reasons"].apply(";".join)
excl = excl.join(cons.set_index("ticker")[["company", "sector"]])
excl.to_csv("outputs/exclusion_log.csv")

included = s.index[s["included"]].tolist()
pd.Series(included, name="ticker").to_csv("outputs/included_tickers.csv", index=False)

print("Included:", len(included), "| Excluded:", len(excl))
print(excl["exclusion_reasons"].value_counts())

Included: 486 | Excluded: 17
exclusion_reasons
unequal_history     10
no_price_data        5
zero_volume_days     2
Name: count, dtype: int64


In [15]:
flags = s.loc[s["included"] & (
    (s["unexplained_big_jumps"] > 0) | (s["unexplained_adj_ratio_changes"] > 0)
), ["unexplained_big_jumps", "unexplained_adj_ratio_changes", "n_splits", "split_dates"]]
flags.to_csv("outputs/review_flags.csv")
print("Included shares with review flags:", len(flags))
print(flags.head(20).to_string())

print("\nSplit traded-value check (ratio should be near 1):")
chk = s.loc[s["included"] & (s["n_splits"] > 0), ["split_dates", "split_traded_value_ratio"]]
print(chk.to_string())

Included shares with review flags: 4
        unexplained_big_jumps  unexplained_adj_ratio_changes  n_splits split_dates
ticker                                                                            
BE                        1.0                            0.0       0.0            
CVNA                      5.0                            0.0       0.0            
DXCM                      1.0                            0.0       1.0  2022-06-13
GL                        1.0                            0.0       0.0            

Split traded-value check (ratio should be near 1):
                             split_dates split_traded_value_ratio
ticker                                                           
AMZN                          2022-06-06                     0.77
ANET               2021-11-18;2024-12-04                1.09;1.08
APH                2021-03-05;2024-06-12                0.93;1.01
AVGO                          2024-07-15                     0.73
BDX              

In [16]:
keep = included + [BENCH]
for name, df in [("adj_close", adj), ("close", close), ("volume", vol), ("dividends", div), ("stock_splits", spl)]:
    df[keep].to_csv(f"data/clean/{name}_valid.csv")

before = cons["sector"].value_counts()
after = cons.set_index("ticker").loc[included, "sector"].value_counts()
print(pd.concat([before.rename("all_503"), after.rename("included")], axis=1).fillna(0).astype(int))

print("\nBenchmark and included shares are on the same", N_REF, "daily dates.")
print("Any NaN left in the clean adjusted close?", bool(adj[keep].isna().any().any()))

                        all_503  included
sector                                   
Industrials                  83        78
Information Technology       76        74
Financials                   76        74
Health Care                  60        58
Consumer Discretionary       47        47
Consumer Staples             33        32
Utilities                    31        30
Real Estate                  30        30
Materials                    24        23
Communication Services       22        20
Energy                       21        20

Benchmark and included shares are on the same 1005 daily dates.
Any NaN left in the clean adjusted close? False
